<a href="https://colab.research.google.com/github/lidchen/ToyTransformer/blob/main/transformer_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

text = "hello world"
chars = sorted(list(set(text)))

stoi = {
    ch:i
    for i,ch in enumerate(chars)
}
itos = {
    i:ch
    for ch,i in stoi.items()
}

data = torch.tensor(
  [stoi[c] for c in text],
  dtype=torch.long
)
def get_batch(B, T):
    ix = torch.randint(
        0,
        len(data) - T - 1,
        (B,)
    )
    x = torch.stack([
        data[i:i+T]
        for i in ix
    ])
    y = torch.stack([
        data[i+1:i+T+1]
        for i in ix
    ])
    return x, y

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

B, T = 4,8
vocab_size = len(chars)
C = 32
lr = 1e-2

embedding = nn.Embedding(vocab_size, C)
linear = nn.Linear(C, vocab_size)

optimizer = torch.optim.Adam(
    list(embedding.parameters()) + list(linear.parameters()),
    lr = lr
)

for step in range (100):
  idx, targets = get_batch(B, T)

  x = embedding(idx)
  logits = linear(x)

  loss = F.cross_entropy(
      logits.view(-1, vocab_size),
      targets.view(-1)
  )
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if step % 10 == 0:
    print(step, loss.item())

s = "h"

idx = torch.tensor([[stoi[c] for c in s]])

x = embedding(idx)

logits = linear(x)

last_logits = logits[0, -1]

probs = F.softmax(last_logits, dim=-1)
print(chars)
print(probs)

next_id = torch.argmax(probs).item()

print(itos[next_id])


0 2.4625608921051025
10 0.6969786286354065
20 0.43387165665626526
30 0.36648404598236084
40 0.3566499352455139
50 0.35355767607688904
60 0.3518606722354889
70 0.3509422838687897
80 0.35038629174232483
90 0.3499264717102051
[' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
tensor([1.6412e-03, 3.1730e-05, 9.9516e-01, 2.4067e-04, 5.4852e-04, 2.3923e-04,
        1.9379e-03, 1.9727e-04], grad_fn=<SoftmaxBackward0>)
e


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

B = 4
T = 8
C = 32
vocab_size = len(chars)
head_size = C
lr = 1e-3

embedding = nn.Embedding(vocab_size, C)
position_embedding = nn.Embedding(T, C)

query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

lm_head = nn.Linear(head_size, vocab_size)

optimizer = torch.optim.Adam(
    list(embedding.parameters()) +
    list(query.parameters()) +
    list(key.parameters()) +
    list(value.parameters()) +
    list(lm_head.parameters()) +
    list(position_embedding.parameters()),
    lr=lr
)

# idx = torch.randint(0, vocab_size, (B, T))

for step in range (1000):
  idx, targets = get_batch()
  x = embedding(idx)
  pos = torch.arange(T)
  pos_emb = position_embedding(pos)
  x = x + pos_emb

  q = query(x)
  k = key(x)
  v = value(x)

  weight = q @ k.transpose(-2, -1)

  # mask
  T_cur = idx.shape[1]
  tril = torch.tril(torch.ones(T_cur, T_cur))
  weight = weight.masked_fill(tril == 0, float('-inf'))

  weight = F.softmax(weight, dim=-1) * (head_size ** -0.5)
  out = x + (weight @ v)
  logits = lm_head(out)

  loss = F.cross_entropy(
    logits.view(-1, vocab_size),
    targets.view(-1)
  )
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if step % 100 == 0:
    print(step, loss.item())

s = "wor"
idx = torch.tensor([[stoi[c] for c in s]])

x = embedding(idx)
q = query(x)
k = key(x)
v = value(x)
weight = q @ k.transpose(-2, -1)

# mask
T_cur = idx.shape[1]
pos = torch.arange(T_cur)
pos_emb = position_embedding(pos)
x = x + pos_emb

tril = torch.tril(torch.ones(T_cur, T_cur))
weight = weight.masked_fill(tril == 0, float('-inf'))

weight = F.softmax(weight, dim=-1) * (head_size ** -0.5)
out = x + (weight @ v)
logits = lm_head(out)

last_logits = logits[0, -1]

probs = F.softmax(last_logits, dim=-1)
torch.set_printoptions(sci_mode=False)

print(chars)
print(probs)

next_id = torch.argmax(probs).item()

print(itos[next_id])


0 2.360994338989258
100 0.912466287612915
200 0.4716312885284424
300 0.31918781995773315
400 0.29050031304359436
500 0.35487380623817444
600 0.1407318264245987
700 0.1447700411081314
800 0.07545362412929535
900 0.08273514360189438
[' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
tensor([0.0001, 0.0001, 0.0001, 0.0009, 0.9966, 0.0006, 0.0002, 0.0015],
       grad_fn=<SoftmaxBackward0>)
l


In [ ]:
import torch
text = "Since the 1830s, the Body Mass Index (BMI) has been used as a standardized metric for assessing health status globally. Originally developed to study average body types among white Europeans, it was not intended to serve as a primary indicator of health. However, due to its simplicity and affordability, BMI has become a cornerstone of healthcare practices worldwide1​. By calculating weight-to-height ratios, BMI simplifies health and fitness levels 1-3​. BMI categorizes individuals as underweight, normal, overweight, and obese 1,2​ (Figure 1) with ideal ranges typically defined as 18.5–25.0 kg/m². The widespread use of BMI has helped simplify healthcare assessments and promote awareness of obesity as a risk factor for several chronic and life-threatening diseases, including diabetes, cardiovascular conditions, hypertension, and certain cancers 1,2,4-8​. Healthcare practitioners often use BMI as a guiding metric for recommending lifestyle changes or interventions. However, this reliance has led to criticism, as BMI fails to account for variations in body composition, genetics, and other factors influencing health. Consequently, individuals whose BMI falls outside the ideal range are often subject to drastic lifestyle recommendations that may not align with their overall health status. Critics argue that diagnosing complex health conditions based solely on weight-to-height ratios is overly simplistic and potentially harmful. For example, BMI does not differentiate between muscle mass and fat, leading to misclassifications, such as labeling muscular individuals as obese. Furthermore, BMI's historical origins and its failure to account for diversity across populations raise questions about its relevance in modern healthcare. This research seeks to evaluate the credibility of BMI as a primary health and fitness indicator by analyzing its strengths and limitations. It also aims to identify a balanced approach, recognizing the utility of BMI while mitigating the harm caused by over-reliance. By exploring recent studies and best practices, this work endeavors to provide a nuanced perspective on how BMI can be integrated into healthcare in a more effective and equitable manner."

# 构建字典
chars = sorted(list(set(text)))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

# 编码
data = torch.tensor([stoi[c] for c in text])

# 采样一个 batch
def get_batch(B, T):
    ix = torch.randint(0, len(data)-T, (B,))
    x = torch.stack([data[i:i+T] for i in ix])
    y = torch.stack([data[i+1:i+T+1] for i in ix])
    return x, y

In [ ]:
from datasets import load_dataset
import torch

ds = load_dataset("roneneldan/TinyStories")
train_ds = ds["train"]
text = ""
for i in range(5000):
    text += train_ds[i]["text"]
chars = sorted(list(set(text)))

stoi = {
    ch:i
    for i,ch in enumerate(chars)
}
itos = {
    i:ch
    for ch,i in stoi.items()
}
all_text = ""

for i in range(5000):
    all_text += train_ds[i]["text"]

data = torch.tensor(
  [stoi[c] for c in all_text],
  dtype=torch.long
)
def get_batch(B, T):
    ix = torch.randint(
        0,
        len(data) - T - 1,
        (B,)
    )
    x = torch.stack([
        data[i:i+T]
        for i in ix
    ])
    y = torch.stack([
        data[i+1:i+T+1]
        for i in ix
    ])
    return x, y

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
from IPython.terminal.embed import embed
import torch
import torch.nn as nn
import torch.nn.functional as F

class toyTransformer(nn.Module):
  def __init__(
      self,
      vocab_size,
      block_size, #T
      embed_dim, #C
      head_size
  ):
    super().__init__()
    self.block_size = block_size
    self.head_size = embed_dim # make embed_dim == head_size for simplicity
    self.token_embedding = nn.Embedding(
        vocab_size,
        embed_dim
    )
    self.position_embedding = nn.Embedding(
        block_size,
        embed_dim
    )
    self.query = nn.Linear(
        embed_dim,
        head_size,
        bias=False
    )
    self.key = nn.Linear(
        embed_dim,
        head_size,
        bias=False
    )
    self.value = nn.Linear(
        embed_dim,
        head_size,
        bias=False
    )
    self.lm_head = nn.Linear(
        embed_dim, # make embed_dim == head_size for simplicity
        vocab_size
    )
    self.ffn = nn.Sequential(
      nn.Linear(embed_dim, 4 * embed_dim),
      nn.ReLU(),
      nn.Linear(4 * embed_dim, embed_dim)
    )
  def forward(self, idx, targets=None):
    B, T = idx.shape
    token_emb = self.token_embedding(idx)
    pos = torch.arange(T, device=idx.device)
    pos_emb = self.position_embedding(pos)
    x = token_emb + pos_emb

    q = self.query(x)
    k = self.key(x)
    v = self.value(x)

    weight = (q @ k.transpose(-2,-1)) * (self.head_size ** -0.5)

    # mask
    T_cur = idx.shape[1]
    tril = torch.tril(torch.ones(T_cur, T_cur, device=idx.device))
    weight = weight.masked_fill(tril == 0, float('-inf'))

    weight = F.softmax(weight, dim=-1)
    out = x + (weight @ v)
    out = out + self.ffn(out)

    logits = self.lm_head(out)
    loss = None
    if targets is not None:
        loss = F.cross_entropy(
            logits.view(-1, logits.shape[-1]),
            targets.view(-1)
        )
    return logits, loss

  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, _ = self(idx)
      last_logits = logits[:, -1, :]
      probs = F.softmax(
        last_logits,
        dim=-1
      )
      next_idx = torch.multinomial(
        probs,
        num_samples=1
      )
      idx = torch.cat(
        [idx, next_idx],
        dim=1
      )
      idx = idx[:, -self.block_size:]
    return idx

In [ ]:
# train a new model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = toyTransformer(
  vocab_size=len(chars),
  block_size=128,
  embed_dim=256,
  head_size=256
)
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=1e-3
)

for step in range(5000):
  idx, targets = get_batch(32, model.block_size)
  idx = idx.to(device)
  targets = targets.to(device)
  logits, loss = model(idx, targets)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 1000 == 0:
    print(loss.item())

torch.save(
  model.state_dict(),
  "model.pt"
)

cuda
4.849401950836182
1.5246204137802124
1.3213121891021729
1.2864466905593872
1.2862521409988403


In [ ]:
# Continue training
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = nn.Module
model.load_state_dict(
  torch.load(
    "model.pt",
    map_location=device
  )
)

model.train()
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=1e-3
)

for step in range(10000):
  idx, targets = get_batch(32, model.block_size)
  idx = idx.to(device)
  targets = targets.to(device)
  logits, loss = model(idx, targets)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 1000 == 0:
    print(loss.item())

torch.save(
  model.state_dict(),
  "model.pt"
)

NameError: name 'torch' is not defined

In [ ]:
model.load_state_dict(
  torch.load(
    "model.pt",
    map_location=device
  )
)
print("model loaded")

model loaded


In [ ]:
model.eval()
start = torch.tensor([[stoi["T"]]])
start = start.to(device)
out = model.generate(
  start,
  max_new_tokens=20000
)
print(out)

decoded = ''.join(
  [itos[i.item()] for i in out[0]]
)
print(decoded)

tensor([[ 1, 70, 48, 66,  1, 70, 65, 56, 67, 52,  1, 62, 68, 67,  1, 70, 52, 65,
         52,  1, 48, 59, 59,  1, 67, 56, 60, 52,  6,  1, 67, 55, 48, 67,  1, 51,
         48, 72,  1, 49, 56, 65, 51,  1, 48,  1, 66, 68, 65, 63, 65, 56, 66, 52,
         51,  8,  1, 41, 55, 52, 72,  1, 66, 55, 52,  1, 63, 48, 65, 67, 56, 52,
         66,  1, 62, 59, 51, 52, 65, 66, 66,  6,  1, 67, 55, 52,  1, 70, 62, 68,
         59, 51,  1, 48, 49, 62, 68, 67,  1, 55, 62, 60, 52,  1, 66, 48, 56, 51,
          6,  1,  3, 30,  1, 67, 62, 68, 50, 55,  1, 67, 55, 52, 72,  1, 51, 52,
         50, 56]], device='cuda:0')
 was write out were all time, that day bird a surprised. They she parties olderss, the would about home said, "I touch they deci
